# Figure 2: PRS Analysis

Multi-panel figure characterizing polygenic risk scores (PRS) derived from bNMF cluster weights.

- **Panel A**: Density curves for each cluster PRS (colored by cluster)
- **Panel B**: Pearson correlation matrix of all PRS columns
- **Panel C**: Forest plot of PRS associations (OR/SD) for T2D, CAD, and Confluence
- **Panel D**: Liability-scale R² bar plot
- **Panel E**: AUC bar plot (train/validation split, EUR and META models)

**Confluence** is defined as: individual has CAD OR T2D (binary 0/1).

All models adjust for age, age², sex, and 16 principal components.

In [ ]:
# --- Setup: Libraries and Paths ---

suppressPackageStartupMessages({
  library(tidyverse)
  library(data.table)
  library(patchwork)
})

# Paths — adjust if running outside AoU workspace
ws <- file.path(Sys.getenv("HOME"), "workspaces", "duplicateofstatinresponse")
project_dir <- file.path(ws, "multiancestry_polygenic")

pheno_path <- file.path(project_dir, "phenotypes", "aou_t2d_cad_phenotypes.csv")
prs_path   <- file.path(project_dir, "results", "prs", "output", "prs_all_clusters.tsv")
output_dir <- file.path(project_dir, "results", "figures")

cat(sprintf("Project directory: %s\n", project_dir))
cat(sprintf("Phenotype path:   %s\n", pheno_path))
cat(sprintf("PRS path:         %s\n", prs_path))

In [ ]:
# --- Source utility and panel scripts ---

script_dir <- file.path(project_dir, "scripts", "analysis")

source(file.path(script_dir, "figure_utils.R"))
source(file.path(script_dir, "figure2_utils.R"))
source(file.path(script_dir, "figure2_panel_a_density.R"))
source(file.path(script_dir, "figure2_panel_b_corr.R"))
source(file.path(script_dir, "figure2_panel_c_forest.R"))
source(file.path(script_dir, "figure2_panel_d_liability.R"))
source(file.path(script_dir, "figure2_panel_e_auc.R"))

cat("All scripts sourced successfully.\n")

In [ ]:
# --- Load and merge data ---

prs_data <- load_prs_data(pheno_path, prs_path)
dat      <- prs_data$dat
prs_meta <- prs_data$prs_meta

cat("\nPRS metadata:\n")
print(prs_meta)

cat("\nSample size by predicted ancestry:\n")
print(dat[, .N, by = ancestry_pred][order(-N)])

In [ ]:
# --- Panel A: PRS Density Curves ---

p_a <- plot_panel_a_density(dat, prs_meta)

options(repr.plot.width = 10, repr.plot.height = 6)
p_a

In [ ]:
# --- Panel B: Pearson Correlation Matrix ---

p_b <- plot_panel_b_corr(dat, prs_meta)

options(repr.plot.width = 10, repr.plot.height = 9)
p_b

In [ ]:
# --- Panel C: Forest Plot (T2D, CAD, Confluence) ---
# This runs 27+ logistic regressions; may take a minute.

p_c <- plot_panel_c_forest(dat, prs_meta)

options(repr.plot.width = 10, repr.plot.height = 14)
p_c

In [ ]:
# --- Panel D: Liability R-squared ---

p_d <- plot_panel_d_liability(dat, prs_meta)

options(repr.plot.width = 10, repr.plot.height = 10)
p_d

In [ ]:
# --- Panel E: AUC Barplot ---
# Splits data 50/50 (seed 47), fits models on training, plots AUC on validation.

p_e <- plot_panel_e_auc(dat, prs_meta)

options(repr.plot.width = 10, repr.plot.height = 10)
p_e

In [ ]:
# --- Assemble Figure ---

layout <- "
AABB
AABB
CCDD
CCDD
CCDD
EEEE
EEEE
"

fig2 <- p_a + p_b + p_c + p_d + p_e +
  plot_layout(design = layout) +
  plot_annotation(
    tag_levels = "A",
    theme = theme(
      plot.tag = element_text(size = 22, face = "bold")
    )
  )

# Save (PNG only)
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

png_path <- file.path(output_dir, "figure2_prs.png")
ggsave(png_path, fig2, width = 18, height = 24, dpi = 300)
cat(sprintf("Saved PNG: %s\n", png_path))

options(repr.plot.width = 18, repr.plot.height = 24)
fig2

## Supplementary Figures

Ancestry-stratified correlation matrices, forest plots, and Nagelkerke R² bar plot.
All saved as standalone PNGs in `results/figures/`.

In [ ]:
# --- Supplementary: Ancestry-stratified Correlation Matrices ---

# Overall (standalone save)
out <- file.path(output_dir, "figure2_corr_overall.png")
ggsave(out, p_b, width = 10, height = 9, dpi = 300)
cat(sprintf("Saved: %s\n", out))

# EUR only
p_b_eur <- plot_panel_b_corr(dat, prs_meta, ancestry_filter = "eur")
out <- file.path(output_dir, "figure2_corr_eur.png")
ggsave(out, p_b_eur, width = 10, height = 9, dpi = 300)
cat(sprintf("Saved: %s\n", out))

# AFR only
p_b_afr <- plot_panel_b_corr(dat, prs_meta, ancestry_filter = "afr")
out <- file.path(output_dir, "figure2_corr_afr.png")
ggsave(out, p_b_afr, width = 10, height = 9, dpi = 300)
cat(sprintf("Saved: %s\n", out))

options(repr.plot.width = 10, repr.plot.height = 9)
p_b_eur

In [ ]:
# --- Supplementary: Ancestry-stratified Forest Plots ---
# Each produces a 3×1 grid (T2D, CAD, Confluence); ~27 regressions each.

# Overall (standalone save)
out <- file.path(output_dir, "figure2_forest_overall.png")
ggsave(out, p_c, width = 10, height = 14, dpi = 300)
cat(sprintf("Saved: %s\n", out))

# EUR only
cat("\nForest plots for EUR individuals...\n")
p_c_eur <- plot_panel_c_forest(dat, prs_meta, ancestry_filter = "eur")
out <- file.path(output_dir, "figure2_forest_eur.png")
ggsave(out, p_c_eur, width = 10, height = 14, dpi = 300)
cat(sprintf("Saved: %s\n", out))

# AFR only
cat("\nForest plots for AFR individuals...\n")
p_c_afr <- plot_panel_c_forest(dat, prs_meta, ancestry_filter = "afr")
out <- file.path(output_dir, "figure2_forest_afr.png")
ggsave(out, p_c_afr, width = 10, height = 14, dpi = 300)
cat(sprintf("Saved: %s\n", out))

options(repr.plot.width = 10, repr.plot.height = 14)
p_c_eur

In [ ]:
# --- Supplementary: Nagelkerke R-squared ---

p_d_nag <- plot_panel_d_liability(dat, prs_meta, metric = "nagelkerke")

out <- file.path(output_dir, "figure2_nagelkerke_r2.png")
ggsave(out, p_d_nag, width = 10, height = 10, dpi = 300)
cat(sprintf("Saved: %s\n", out))

options(repr.plot.width = 10, repr.plot.height = 10)
p_d_nag

In [ ]:
# --- Export regression results as supplementary tables ---

table_dir <- file.path(project_dir, "results", "tables")
dir.create(table_dir, recursive = TRUE, showWarnings = FALSE)

# OR table from Panel C
cat("Re-running regressions for table export...\n")
or_results <- run_all_regressions(dat, prs_meta)
or_path <- file.path(table_dir, "figure2_or_results.csv")
fwrite(or_results, or_path)
cat(sprintf("OR results saved to: %s\n", or_path))

# Liability R-squared table from Panel D
r2_results <- run_all_liability_r2(dat, prs_meta)
r2_path <- file.path(table_dir, "figure2_liability_r2_results.csv")
fwrite(r2_results, r2_path)
cat(sprintf("Liability R-squared results saved to: %s\n", r2_path))

cat("\n--- OR Results ---\n")
print(or_results[, .(outcome_label, label, gwas_ancestry, OR, CI_lo, CI_hi, p_value, n_cases, n_total)])

cat("\n--- Liability R-squared Results ---\n")
print(r2_results[, .(outcome_label, label, gwas_ancestry, r2_obs, r2_liability, prevalence)])